In [34]:
import time

from lerobot.processor import make_default_processors

from lerobot.teleoperators.so101_leader.config_so101_leader import SO101LeaderConfig
from lerobot.teleoperators.so101_leader.so101_leader import SO101Leader
from lerobot.teleoperators.bi_so100_leader import BiSO100Leader, BiSO100LeaderConfig

from lerobot.utils.robot_utils import busy_wait
from lerobot.utils.visualization_utils import init_rerun, log_rerun_data
from lerobot.robots.xlerobot import XLerobotClientConfig, XLerobotClient
from lerobot.cameras.configs import CameraConfig, Cv2Rotation, ColorMode
from lerobot.cameras.realsense import RealSenseCamera, RealSenseCameraConfig

FPS = 30

In [41]:
camera_config = {
            "head": RealSenseCameraConfig(
            serial_number_or_name="141722076677",  # Replace with camera SN
            fps=30,
            width=1280,
            height=720,
            color_mode=ColorMode.BGR, # Request BGR output
            rotation=Cv2Rotation.NO_ROTATION,
            use_depth=False
        )
}

# Initialize the robot and teleoperator config

follower_config = XLerobotClientConfig(remote_ip = '10.16.116.39', cameras = camera_config)

# Initialize the robot and teleoperator
follower = XLerobotClient(follower_config)

# Connect to the robot and teleoperator
follower.connect()

In [30]:
leader_config = BiSO100LeaderConfig(left_arm_port="/dev/ttyACM0", right_arm_port= "/dev/ttyACM1", id="leader_arm")
leader = BiSO100Leader(leader_config)
leader.connect()

In [31]:
def transform_arm_keys(original_dict: dict) -> dict:
    """
    Преобразует ключи формата left_* и right_* в left_arm_* и right_arm_*
    """
    transformed = {}
    
    for key, value in original_dict.items():
        if key.startswith('left_'):
            new_key = key.replace('left_', 'left_arm_', 1)
        elif key.startswith('right_'):
            new_key = key.replace('right_', 'right_arm_', 1)
        else:
            new_key = key  # Оставляем без изменений
            
        transformed[new_key] = value
    
    return transformed

In [ ]:
follower.get_observation()['head']

(720, 1280, 3)

In [38]:
follower.disconnect()

In [17]:
obs = follower.get_observation()
del obs['observation.state']
del obs['head']
obs

{'left_arm_shoulder_pan.pos': -1.7764618800888172,
 'left_arm_shoulder_lift.pos': -91.26712328767124,
 'left_arm_elbow_flex.pos': 99.72665148063783,
 'left_arm_wrist_flex.pos': 43.81420276909333,
 'left_arm_wrist_roll.pos': -3.539355520338077,
 'left_arm_gripper.pos': 0.06934812760055478,
 'right_arm_shoulder_pan.pos': -6.39083856667898,
 'right_arm_shoulder_lift.pos': -93.33614508646141,
 'right_arm_elbow_flex.pos': 100.0,
 'right_arm_wrist_flex.pos': 68.8334050796384,
 'right_arm_wrist_roll.pos': 1.4323607427055691,
 'right_arm_gripper.pos': 0.0,
 'head_motor_1.pos': 80.31060805475124,
 'head_motor_2.pos': -17.519685039370074,
 'x.vel': 0.0,
 'y.vel': 0.0,
 'theta.vel': 0.0}

In [23]:
action['head_motor_1.pos'] = obs['head_motor_1.pos']
action['head_motor_2.pos'] = -25
base_action = {"x.vel": 0.,
            "y.vel": 0.,
            "theta.vel": 0.}
action = {**action, **base_action}
action

{'left_shoulder_pan.pos': -5.321100917431181,
 'left_shoulder_lift.pos': -100.0,
 'left_elbow_flex.pos': 96.25912408759123,
 'left_wrist_flex.pos': 71.40396210163652,
 'left_wrist_roll.pos': -0.5872931126534979,
 'left_gripper.pos': 0.49627791563275436,
 'right_shoulder_pan.pos': 7.059333044608067,
 'right_shoulder_lift.pos': -99.40903334740396,
 'right_elbow_flex.pos': 94.95644199908298,
 'right_wrist_flex.pos': 71.11383108935127,
 'right_wrist_roll.pos': -0.3427366200896387,
 'right_gripper.pos': 0.741962077493817,
 'head_motor_1.pos': 80.31060805475124,
 'head_motor_2.pos': -25,
 'x.vel': 0.0,
 'y.vel': 0.0,
 'theta.vel': 0.0}

In [27]:
follower._state_order

('left_arm_shoulder_pan.pos',
 'left_arm_shoulder_lift.pos',
 'left_arm_elbow_flex.pos',
 'left_arm_wrist_flex.pos',
 'left_arm_wrist_roll.pos',
 'left_arm_gripper.pos',
 'right_arm_shoulder_pan.pos',
 'right_arm_shoulder_lift.pos',
 'right_arm_elbow_flex.pos',
 'right_arm_wrist_flex.pos',
 'right_arm_wrist_roll.pos',
 'right_arm_gripper.pos',
 'head_motor_1.pos',
 'head_motor_2.pos',
 'x.vel',
 'y.vel',
 'theta.vel')

In [26]:
follower.send_action(action)

{'left_arm_shoulder_pan.pos': np.float32(0.0),
 'left_arm_shoulder_lift.pos': np.float32(0.0),
 'left_arm_elbow_flex.pos': np.float32(0.0),
 'left_arm_wrist_flex.pos': np.float32(0.0),
 'left_arm_wrist_roll.pos': np.float32(0.0),
 'left_arm_gripper.pos': np.float32(0.0),
 'right_arm_shoulder_pan.pos': np.float32(0.0),
 'right_arm_shoulder_lift.pos': np.float32(0.0),
 'right_arm_elbow_flex.pos': np.float32(0.0),
 'right_arm_wrist_flex.pos': np.float32(0.0),
 'right_arm_wrist_roll.pos': np.float32(0.0),
 'right_arm_gripper.pos': np.float32(0.0),
 'head_motor_1.pos': np.float32(80.31061),
 'head_motor_2.pos': np.float32(-25.0),
 'x.vel': np.float32(0.0),
 'y.vel': np.float32(0.0),
 'theta.vel': np.float32(0.0),
 'action': array([  0.     ,   0.     ,   0.     ,   0.     ,   0.     ,   0.     ,
          0.     ,   0.     ,   0.     ,   0.     ,   0.     ,   0.     ,
         80.31061, -25.     ,   0.     ,   0.     ,   0.     ],
       dtype=float32)}

In [4]:
follower.disconnect()

In [17]:
action = leader.get_action()
action = {f"left_arm_{k}": v for k, v in action.items()}
action["head_motor_1.pos"] = 0.
action["head_motor_2.pos"] = 0.
follower.send_action(action)

{'left_arm_shoulder_pan.pos': np.float32(-6.495413),
 'left_arm_shoulder_lift.pos': np.float32(-100.0),
 'left_arm_elbow_flex.pos': np.float32(96.076645),
 'left_arm_wrist_flex.pos': np.float32(71.834625),
 'left_arm_wrist_roll.pos': np.float32(-5.65937),
 'left_arm_gripper.pos': np.float32(29.528536),
 'head_motor_1.pos': np.float32(0.0),
 'head_motor_2.pos': np.float32(0.0),
 'action': array([  -6.495413, -100.      ,   96.076645,   71.834625,   -5.65937 ,
          29.528536,    0.      ,    0.      ], dtype=float32)}

In [14]:
follower.send_action(action)

{'left_arm_shoulder_pan.pos': np.float32(-6.495413),
 'left_arm_shoulder_lift.pos': np.float32(-100.0),
 'left_arm_elbow_flex.pos': np.float32(95.89416),
 'left_arm_wrist_flex.pos': np.float32(72.35142),
 'left_arm_wrist_roll.pos': np.float32(-5.7127604),
 'left_arm_gripper.pos': np.float32(21.009098),
 'head_motor_1.pos': np.float32(0.0),
 'head_motor_2.pos': np.float32(0.0),
 'action': array([  -6.495413 , -100.       ,   95.89416  ,   72.35142  ,
          -5.7127604,   21.009098 ,    0.       ,    0.       ],
       dtype=float32)}

In [5]:
follower.disconnect()

### Keyboard teleop

In [4]:
from lerobot.teleoperators.keyboard.teleop_keyboard import KeyboardTeleop, KeyboardTeleopConfig
import time
import numpy as np

In [2]:
#Init the keyboard instance
keyboard_config = KeyboardTeleopConfig()
keyboard = KeyboardTeleop(keyboard_config)
keyboard.connect()